<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson05/01_gpt2_on_the_hub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 05 · Notebook 1: Our Shakespeare GPT in 🤗 Transformers, and on the Hub

**Hugging Face** is the "GitHub of machine learning". Its **Hub** ([huggingface.co](https://huggingface.co)) hosts millions of models and datasets that anyone can download, and it also makes the Python libraries most people use to work with them:

| library | what it does | notebook |
|---|---|---|
| `transformers` | model classes (GPT-2, BERT, Llama, ...), downloading and running models | 1, 3 |
| `huggingface_hub` | talking to the Hub: logging in, uploading, downloading | 1 |
| `datasets` | loading, filtering and sharing datasets | 2 |
| `tokenizers` | fast tokenizers, including training your own | 2 |

Last week we downloaded GPT-2 with `GPT2LMHeadModel.from_pretrained("gpt2")`, rebuilt it in ~60 lines of our own PyTorch, and trained a mini GPT on Shakespeare. Today we train the **same mini GPT**, but with the `GPT2LMHeadModel` class from `transformers` instead of our own. Then we put it on the Hub, so that anyone (e.g. your neighbour) can download it with one line.

**What you will do:**
- create a GPT-2 of any size with `GPT2Config`, and check that it is the same model as our class from last week
- train it on Shakespeare with the same 5-step training loop
- generate text with `model.generate()`
- save the model, log in to Hugging Face and upload it to your account
- download a model from the Hub and run it

**Contents:** 0. Setup · 1. The data · 2. A GPT-2 of any size · 3. Forward pass and loss · 4. Training · 5. Generating text · 6. Saving the model · 7. Logging in to Hugging Face · 8. Uploading to the Hub · 9. Downloading from the Hub · 10. Summary

**Before you start:**
- *Runtime → Change runtime type → T4 GPU* (for the training in section 4).
- You need your Hugging Face account from Assignment 02.

---
## 0. Setup

`transformers` and `huggingface_hub` are already installed on Colab.

In [ ]:
import math
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import transformers

print("transformers", transformers.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

---
## 1. The data (the same as last week)

Tiny Shakespeare, 1 MB of text. As last week, our tokens are single **characters**, so the vocabulary has 65 "tokens".

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")
with open("input.txt", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}     # character -> id
itos = {i: ch for i, ch in enumerate(chars)}     # id -> character

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]        # first 90% for training, the last 10% for validation
print(f"{len(text):,} characters, vocabulary of {vocab_size}")
print(encode("To be, or not to be"))

One change against last week: `get_batch` now returns **only the input** `x`, windows of 128 characters. Last week we also built the target `y`, the same window shifted by one. Hugging Face models do the shifting for us (section 3).

In [ ]:
context = 128
batch_size = 64

def get_batch(split):
    d = train_data if split == "train" else val_data
    starts = torch.randint(len(d) - context, (batch_size,))
    x = torch.stack([d[s:s + context] for s in starts])
    return x.to(device)

x = get_batch("train")
print(x.shape)
print(repr(decode(x[0, :60].tolist())))

---
## 2. A GPT-2 of any size: `GPT2Config`

Last week, `from_pretrained("gpt2")` gave us GPT-2 **with its trained weights**. Now we want a GPT-2 with **random weights** and our own size. In `transformers`, a model is made from two things:

- a **config**: the recipe (how many blocks, how wide, how many heads, ...)
- a **model class** that builds the network from the config: `GPT2LMHeadModel`

We use the same sizes as our mini GPT from last week:

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=vocab_size,   # 65 characters (GPT-2 has 50,257 tokens)
    n_positions=context,     # the longest text the model can see: 128 (GPT-2: 1,024)
    n_embd=128,              # vector size d (GPT-2: 768)
    n_layer=4,               # number of blocks (GPT-2: 12)
    n_head=4,                # attention heads in each block (GPT-2: 12)
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,   # no dropout, like our class last week
    bos_token_id=None, eos_token_id=None,               # GPT-2's <|endoftext|> (id 50256) does not exist in our 65 characters
)
config

The config has many more settings than we gave it. Everything we did not set keeps GPT-2's default value. (*Dropout* randomly switches off some numbers during training, which helps against overfitting. Our class from last week did not have it, so we switched it off.)

Now the model itself:

In [ ]:
torch.manual_seed(1337)
model = GPT2LMHeadModel(config).to(device)
model

*(You may see a warning about `loss_type=None`. You can ignore it.)*

Compare this with GPT-2 small's printout from last week: the same structure, only smaller numbers (`Embedding(65, 128)` instead of `Embedding(50257, 768)`, 4 blocks instead of 12).

**Look inside.** Is it really the same network as our `GPT` class from last week? Let's count the parameters:

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())

print(f"parameters: {count(model):,}")
for name, p in model.named_parameters():
    if ".h." not in name or ".h.0." in name:        # all blocks look the same, so we show only block 0
        print(f"   {name:40} {str(tuple(p.shape)):12} {p.numel():>8,}")

**818,048 parameters**, exactly as many as our `mini` model last week. The layers just have different names:

| our `GPT` (lesson 04) | `GPT2LMHeadModel` |
|---|---|
| `wte`, `wpe` | `transformer.wte`, `transformer.wpe` |
| `blocks[i].ln_1`, `ln_2` | `transformer.h[i].ln_1`, `ln_2` |
| `blocks[i].qkv`, `proj` | `transformer.h[i].attn.c_attn`, `attn.c_proj` |
| `blocks[i].fc`, `fc_out` | `transformer.h[i].mlp.c_fc`, `mlp.c_proj` |
| `ln_f` | `transformer.ln_f` |
| `x @ wte.weight.T` (unembedding) | `lm_head` |

Where are the parameters of `lm_head`? It **shares** its weights with `wte` (*weight tying*, section 8 of last week), so they are counted only once:

In [ ]:
print(model.lm_head.weight is model.transformer.wte.weight)

---
## 3. Forward pass and loss

A Hugging Face model returns an **output object**. Besides the `logits`, it can also compute the **loss** for us, if we give it `labels`:

In [ ]:
x = get_batch("train")
out = model(x, labels=x)

print(type(out).__name__)
print("logits:", out.logits.shape)                   # (batch, tokens, vocab_size), the same as our model's output
print(f"loss: {out.loss.item():.3f}    ln(vocab_size) = {math.log(vocab_size):.3f}")

Notice `labels=x`: the labels are **the same** as the input, not shifted! The model shifts them by itself: the scores at position *t* are compared with the character at position *t + 1*. Let's check this by computing the loss ourselves, as we did last week:

In [ ]:
logits = out.logits[:, :-1]          # predictions from positions 0 ... 126
targets = x[:, 1:]                   # the characters at positions 1 ... 127
our_loss = F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))
print(f"HF loss: {out.loss.item():.4f}   our loss: {our_loss.item():.4f}")

The same number. ⚠️ A classic bug: if you shift the labels yourself *and* pass them as `labels`, they get shifted twice, and the model learns to predict the character **two** places ahead.

The loss before training is about ln(65) ≈ 4.17: the model gives every character about the same chance.

---
## 4. Training

The **same 5 steps** as in lesson 2 and lesson 4. The only change: the loss comes from `model(x, labels=x).loss`. Every 250 steps we measure the training and validation loss.

In [ ]:
@torch.no_grad()
def estimate_loss(model, n_batches=20):
    # Average loss on a few random batches, for training and validation data.
    result = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(n_batches):
            x = get_batch(split)
            losses.append(model(x, labels=x).loss.item())
        result[split] = sum(losses) / len(losses)
    return result

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
steps = 3000 if device == "cuda" else 300        # on a CPU only a short demo
history = []
t0 = time.time()

for step in range(1, steps + 1):
    x = get_batch("train")
    loss = model(x, labels=x).loss       # 1. FORWARD + 2. LOSS
    optimizer.zero_grad()                # 5. RESET
    loss.backward()                      # 3. BACKWARD
    optimizer.step()                     # 4. UPDATE

    if step % 250 == 0 or step == steps:
        losses = estimate_loss(model)
        history.append((step, losses["train"], losses["val"]))
        print(f"step {step:5}:  train loss {losses['train']:.3f}   validation loss {losses['val']:.3f}   ({time.time() - t0:.0f} s)")

In [ ]:
done, train_losses, val_losses = zip(*history)
plt.plot(done, train_losses, label="training")
plt.plot(done, val_losses, label="validation")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

---
## 5. Generating text with `model.generate()`

Last week we wrote the generation loop ourselves. `transformers` models have it built in: `generate()` takes the prompt as a tensor of token ids, of shape `[1, T]`.

In [ ]:
prompt = "ROMEO:\n"
ids = torch.tensor([encode(prompt)], device=device)

torch.manual_seed(42)
out = model.generate(ids, max_new_tokens=100, do_sample=True, temperature=0.8)
print(decode(out[0].tolist()))

- `do_sample=True` draws the next character at random according to its probability, `do_sample=False` always takes the most likely one (*greedy*).
- `temperature` works as last week: lower = safer and more boring, higher = wilder.

**Try it:** Other prompts (only characters that appear in Shakespeare!), `do_sample=False`, `temperature=0.5` and `1.5`.

**Look inside: the context limit.** Our model has position embeddings only for positions 0 ... 127 (`n_positions=128`), so it can never see more than 128 characters. Last week our own `generate` function always showed the model only the **last** 128 characters. `model.generate()` does not do that: it always feeds the whole text. So the prompt plus `max_new_tokens` must stay within 128 characters. If you ask for more, the model looks for position 128 in a table that ends at 127, and you get an `IndexError` (on a GPU, a `device-side assert` error, after which you must restart the runtime).

Every GPT has such a limit, the **context window**: 1,024 tokens for GPT-2, hundreds of thousands for today's chatbots. For longer texts we use our sliding window from last week, now with `.logits`:

In [ ]:
@torch.no_grad()
def generate_long(model, ids, n_new, temperature=0.8):
    # Like last week: the model always sees only the last `context` characters.
    for _ in range(n_new):
        logits = model(ids[:, -context:]).logits[:, -1, :]
        probs = F.softmax(logits / temperature, dim=-1)
        ids = torch.cat([ids, torch.multinomial(probs, num_samples=1)], dim=1)
    return ids

torch.manual_seed(42)
print(decode(generate_long(model, ids, 500)[0].tolist()))

---
## 6. Saving the model

`save_pretrained` writes the model into a folder, in the same format as every model on the Hub:

In [ ]:
model.save_pretrained("shakespeare-gpt")
!ls -l shakespeare-gpt

- `config.json`: the recipe from section 2, as text
- `model.safetensors`: the weights. 818,048 numbers × 4 bytes ≈ 3.3 MB
- `generation_config.json`: default settings for `generate()`

Let's look at the config:

In [ ]:
!cat shakespeare-gpt/config.json

One thing is missing: the **vocabulary**. Without it, nobody knows that id 0 is `"\n"` and id 13 is `"A"`. We save our 65 characters into a small text file next to the model (in notebook 2 we do this properly, with a real tokenizer):

In [ ]:
with open("shakespeare-gpt/chars.txt", "w", encoding="utf-8") as f:
    f.write("".join(chars))
!ls -l shakespeare-gpt

---
## 7. Logging in to Hugging Face

To upload anything to the Hub, Hugging Face needs to know who you are. On a remote computer (Colab, a server, a GitHub Action) you cannot click "log in" in a browser, so instead you use an **access token**: a long random password that starts with `hf_`.

**1. Create a token:** [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → *Create new token* → *Write* → copy it. (If you kept the token from Assignment 02, you can reuse it.)

**2. Store it in Colab Secrets** (recommended):
- click the 🔑 icon in the left sidebar → *Add new secret*
- Name: `HF_TOKEN`, Value: your token
- switch on *Notebook access*

The `huggingface_hub` library finds the `HF_TOKEN` secret automatically, in every notebook you open. You never have to paste it again, and last week's warning about a missing `HF_TOKEN` goes away.

⚠️ **A token is a password.** Never write it into a notebook cell: notebooks get shared and pushed to GitHub, and bots scan GitHub for tokens within minutes. If a token leaks, delete it on the tokens page.

Let's check who we are. If there is no secret, `login()` asks for the token, the same way as in Assignment 02:

In [ ]:
from huggingface_hub import HfApi, login, whoami

try:
    user = whoami()["name"]
except Exception:            # no HF_TOKEN secret: paste a token with "Write" access
    login()
    user = whoami()["name"]
print("logged in as", user)

*(Colab may ask whether this notebook can access the `HF_TOKEN` secret: click *Grant access*.)*

---
## 8. Uploading to the Hub

Every model on the Hub lives in a **repository**, `username/model-name`, which is a git repository with a web page. `push_to_hub` creates the repository and uploads `config.json` and the weights:

In [ ]:
repo_id = f"{user}/shakespeare-gpt"
model.push_to_hub(repo_id)            # add private=True if you do not want others to see it
print(f"https://huggingface.co/{repo_id}")

`push_to_hub` uploads only what `transformers` knows about. Other files we upload with `HfApi`, as in Assignment 02: our vocabulary, and a **model card** `README.md`, the front page of the repository. The part between the `---` lines is metadata, which the Hub uses for search and for the widgets on the page.

In [ ]:
api = HfApi()
api.upload_file(path_or_fileobj="shakespeare-gpt/chars.txt", path_in_repo="chars.txt", repo_id=repo_id)

final = history[-1]
model_card = f'''---
library_name: transformers
pipeline_tag: text-generation
language: en
---

# Shakespeare GPT

A tiny GPT-2 ({count(model):,} parameters) trained from scratch on
[tiny Shakespeare](https://github.com/karpathy/char-rnn/tree/master/data/tinyshakespeare),
one **character** per token. Made in the [Data Science Practicum](https://github.com/simecek/dspracticum2026), lesson 05.

- {config.n_layer} blocks, vector size {config.n_embd}, {config.n_head} heads, context of {config.n_positions} characters
- {final[0]} training steps, validation loss {final[2]:.3f}

The vocabulary is in `chars.txt`: the character with id `i` is `chars[i]`.
'''
api.upload_file(path_or_fileobj=model_card.encode("utf-8"), path_in_repo="README.md", repo_id=repo_id)
print(f"https://huggingface.co/{repo_id}")

Open the link. On the page, look at:
- the **model card** we just wrote
- **Files and versions**: `config.json`, `model.safetensors`, `chars.txt`. Under *History* you see our three uploads as three git **commits**
- the model size, shown next to the files

---
## 9. Downloading from the Hub

Now the other direction. This is exactly what `from_pretrained("gpt2")` did last week: it downloads `config.json` and the weights, builds the model from the config and loads the weights into it.

In [ ]:
from huggingface_hub import hf_hub_download

repo = repo_id      # try your neighbour's model, e.g. "their-username/shakespeare-gpt"

downloaded = GPT2LMHeadModel.from_pretrained(repo).to(device)
with open(hf_hub_download(repo, "chars.txt"), encoding="utf-8") as f:
    their_chars = f.read()

def decode_with(chars, ids):
    return "".join(chars[i] for i in ids)

ids = torch.tensor([[their_chars.index(ch) for ch in "JULIET:\n"]], device=device)
torch.manual_seed(0)
print(decode_with(their_chars, generate_long(downloaded, ids, 300)[0].tolist()))

**Try it:**
- Ask your neighbour for their username and download their model. Whose model writes better Shakespeare?
- Find Hugging Face's cache folder: `!ls ~/.cache/huggingface/hub`. Downloaded models are stored there, so the second `from_pretrained` is instant.

**Why not `pipeline("text-generation", model=repo)`?** A `pipeline` needs a **tokenizer** stored next to the model, in a format that `transformers` understands. Our `chars.txt` is our own invention, so the pipeline does not know what to do with it. In **notebook 2** we train a real tokenizer with the `tokenizers` library, and then our model works with `pipeline` like any other model on the Hub.

---
## 10. Summary

| step | code |
|---|---|
| a model of any size | `config = GPT2Config(...)`, `model = GPT2LMHeadModel(config)` |
| loss | `model(x, labels=x).loss` (the model shifts the labels itself) |
| generating | `model.generate(ids, max_new_tokens=..., do_sample=True, temperature=...)` |
| saving | `model.save_pretrained("folder")` |
| logging in | the `HF_TOKEN` secret in Colab, or `login()` |
| uploading | `model.push_to_hub("user/name")`, `HfApi().upload_file(...)` |
| downloading | `GPT2LMHeadModel.from_pretrained("user/name")`, `hf_hub_download(repo, file)` |

**New words:** Hugging Face Hub, repository, config, `from_pretrained` / `save_pretrained` / `push_to_hub`, safetensors, access token, Colab Secrets, model card, context window.

### Exercises
1. Train a bigger model (e.g. `n_embd=192, n_layer=6, n_head=6`) and upload it as a second repository. Write the validation loss of both models into their model cards.
2. Switch dropout on (`resid_pdrop=0.1, embd_pdrop=0.1, attn_pdrop=0.1`). How do the training and validation loss change?
3. Make one of your repositories private (`private=True`, or *Settings* on the model page). Can your neighbour still download it? Can you?

### Further reading
- [Hugging Face Hub documentation](https://huggingface.co/docs/hub): repositories, model cards, security tokens
- [Hugging Face LLM course](https://huggingface.co/learn/llm-course): chapter 4 is about sharing models on the Hub